# A training-only experiment for a higher-quality 200k diabetes model

## 1. Introduction

This notebook asks whether a more structured synthetic-data method can improve the current M1 screening model. It uses the project’s unbalanced real-data CSV, creates synthetic diabetes profiles using sequential CART sampling, and evaluates one predeclared candidate.

1. Confirm the feature lists saved in M1 and M2, then use their shared features throughout this experiment.
2. Remove any records from the unbalanced source that exactly match the fixed test respondents. This is essential because the source CSV contains the test rows.
3. Compare a real-only, balanced logistic-regression reference and the 200k CART-synthetic recipe using nested, stratified cross-validation on the remaining real rows.
4. Inspect whether the generated profiles resemble held-out real diabetes profiles using a real-versus-synthetic classifier and feature summaries.
5. After the method and threshold are fixed using training-only results, make one final comparison on the same untouched-label test set used by M1 and M2.

**Source tables/files:** the repository export of `diabetes_final_clean_merged_unbalanced`, `diabetes_model_train`, `diabetes_model_test`, and the saved M1/M2 model bundles. No live database connection is needed because the DBeaver-exported CSV is already present.

**Outputs:** verified model feature lists; eligible real-row counts; synthetic-fidelity diagnostics; nested CV results; one final test comparison; and a CSV report under `results/`. The fitted experiment model is not saved into `application/models/`.

## 2. Experimental safeguards and decision rule

The **test labels are not used** to build synthetic records, fit preprocessing, choose a generator, or set a threshold. The fixed test rows are removed from the larger source file by exact full-row matching because they are present in that source. In cross-validation, held-out fold rows are used only to block exact synthetic copies by profile; their labels are used only for validation metrics.

The generator and feature list are fixed before scoring. The test set is opened for scoring only after cross-validation is complete. Recall is the primary requirement (at least 0.85); among models that meet it, specificity is the main comparison, with balanced-test precision and an approximate 14%-prevalence precision also reported. The approximate adjustment is not an individual probability.

The candidate uses a sequential CART method: for each M1/M2 feature in order, a small decision tree estimates that answer from previously generated answers, then samples from the tree’s class probabilities. This preserves observed answer categories and some conditional relationships; it does not guarantee that the synthetic people are realistic.

## 3. Imports and shared metrics

Load only the tools needed for data checks, the fixed logistic-regression pipeline, cross-validation, fidelity diagnostics, and final scoring. All random procedures use seed 42.

In [1]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

TARGET = "Diabetes_binary"
RANDOM_STATE = 42
TARGET_RECALL = 0.85
REAL_PREVALENCE = 0.14
REAL_NEGATIVE_CAP = 100_000
BALANCED_TARGET_ROWS = 200_000
SYNTHETIC_SEED = 42

ROOT = Path("../..")
DATA = ROOT / "data"
MODELS = ROOT / "application" / "models"
RESULTS = Path("results")
RESULTS.mkdir(parents=True, exist_ok=True)

NON_BINARY = {"BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Income"}

def make_logreg(features):
    """Match the saved project pipeline: scale non-binary inputs inside training."""
    scaled = [name for name in features if name in NON_BINARY]
    prep = ColumnTransformer(
        [("scale", StandardScaler(), scaled)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(C=np.inf, max_iter=2000)),
    ])

def threshold_for(y_true, probabilities, goal=TARGET_RECALL):
    """Select the highest threshold reaching the recall goal on training OOF scores."""
    _, recall, thresholds = precision_recall_curve(y_true, probabilities)
    eligible = thresholds[recall[:-1] >= goal]
    if len(eligible) == 0:
        raise ValueError(f"No threshold reaches the requested recall goal ({goal:.0%}).")
    return float(eligible.max())

def evaluate(y_true, probabilities, threshold):
    """Calculate classification and probability metrics at one fixed threshold."""
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, predicted),
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "Precision (balanced test)": tp / (tp + fp) if tp + fp else np.nan,
        "F1": f1_score(y_true, predicted, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, probabilities),
        "Brier score": brier_score_loss(y_true, probabilities),
        "False negatives": fn,
        "False positives": fp,
        "Share flagged (balanced test)": float(predicted.mean()),
    }

def prevalence_adjustment(recall, specificity, prevalence=REAL_PREVALENCE):
    """Approximate precision and share flagged at the assumed population prevalence."""
    flagged_share = recall * prevalence + (1 - specificity) * (1 - prevalence)
    precision = (recall * prevalence / flagged_share) if flagged_share else np.nan
    return precision, flagged_share

print(f"scikit-learn: {sklearn.__version__}")

scikit-learn: 1.9.1


## 4. Load saved bundles and verify the final model features

The current saved artifacts are authoritative; older notes in the repository listed stale feature counts. We read the feature names and order from both bundles and stop if the lists differ, so the experiment cannot silently compare different inputs.

In [2]:
m1_bundle = joblib.load(MODELS / "logreg_70k.joblib")
m2_bundle = joblib.load(MODELS / "logreg_200k.joblib")
FEATURES_M1 = list(m1_bundle["features"])
FEATURES_M2 = list(m2_bundle["features"])

feature_check = pd.DataFrame([
    {"Model": "M1: saved 70k real model", "Feature count": len(FEATURES_M1),
     "Features in saved order": ", ".join(FEATURES_M1), "Threshold": m1_bundle["threshold"]},
    {"Model": "M2: saved 200k synthetic model", "Feature count": len(FEATURES_M2),
     "Features in saved order": ", ".join(FEATURES_M2), "Threshold": m2_bundle["threshold"]},
])
display(feature_check.style.hide(axis="index"))

if FEATURES_M1 != FEATURES_M2:
    raise ValueError("M1 and M2 feature names/order differ. Resolve that before this aligned experiment.")
FEATURES = FEATURES_M1
print(f"Confirmed: both models use the same {len(FEATURES)} features in the same order.")

Model,Feature count,Features in saved order,Threshold
M1: saved 70k real model,14,"HighBP, HighChol, BMI, Stroke, HeartDiseaseorAttack, Fruits, HvyAlcoholConsump, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Income",0.396314
M2: saved 200k synthetic model,14,"HighBP, HighChol, BMI, Stroke, HeartDiseaseorAttack, Fruits, HvyAlcoholConsump, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Income",0.375532


Confirmed: both models use the same 14 features in the same order.


## 5. Load CSVs and remove held-out test rows from the source export

The unbalanced source export includes the fixed test rows. We remove exact full-row matches before any split, fit, or synthetic-data generation. The test file is only used here for this overlap removal and later for the one final score. The resulting training source should contain 187,357 real no-diabetes rows and 28,078 real diabetes rows if the checked-in exports match the project’s current split.

In [3]:
source_path = DATA / "03_clean+merged_unbalanced(cmu)" / "diabetes_final_clean_merged_unbalanced.csv"
train_path = DATA / "05_cmb_train_test_for_models" / "diabetes_model_train.csv"
test_path = DATA / "05_cmb_train_test_for_models" / "diabetes_model_test.csv"

source_raw = pd.read_csv(source_path)
fixed_train = pd.read_csv(train_path)
fixed_test = pd.read_csv(test_path)

if source_raw.columns.tolist() != fixed_test.columns.tolist():
    raise ValueError("Source and fixed test columns/order differ; inspect the exports before proceeding.")
if source_raw.isna().any().any() or fixed_test.isna().any().any():
    raise ValueError("Missing values found; resolve data quality before fitting this experiment.")

# Hash complete rows so no fixed-test respondent remains in the larger source export.
source_hash = pd.util.hash_pandas_object(source_raw, index=False)
test_hashes = set(pd.util.hash_pandas_object(fixed_test, index=False).tolist())
keep_source = ~source_hash.isin(test_hashes)
real_pool = source_raw.loc[keep_source].drop_duplicates().reset_index(drop=True)
remaining_hashes = set(pd.util.hash_pandas_object(real_pool, index=False).tolist())
overlap_after_removal = len(remaining_hashes & test_hashes)

source_counts = real_pool[TARGET].value_counts().sort_index()
source_summary = pd.DataFrame({
    "Check": ["Rows in source export", "Fixed test rows", "Rows removed as exact test matches",
              "Eligible real rows", "Eligible no-diabetes rows", "Eligible diabetes rows",
              "Exact test-row overlap remaining"],
    "Count": [len(source_raw), len(fixed_test), int(keep_source.size - keep_source.sum()),
              len(real_pool), int(source_counts.get(0, 0)), int(source_counts.get(1, 0)),
              overlap_after_removal],
})
display(source_summary.style.hide(axis="index"))

if overlap_after_removal != 0:
    raise ValueError("Exact held-out test rows remain in the training source.")
if source_counts.get(0, 0) < REAL_NEGATIVE_CAP:
    raise ValueError("Not enough eligible real no-diabetes rows for the planned 100k recipe.")
if fixed_test[TARGET].nunique() != 2:
    raise ValueError("The fixed test file must contain both target classes.")
print("Source/test overlap check passed; no fixed test respondent is available for fitting.")

Check,Count
Rows in source export,229474
Fixed test rows,14039
Rows removed as exact test matches,14039
Eligible real rows,215435
Eligible no-diabetes rows,187357
Eligible diabetes rows,28078
Exact test-row overlap remaining,0


Source/test overlap check passed; no fixed test respondent is available for fitting.


## 6. Validate the aligned data and describe the 200k recipe

The prediction model will use only the shared M1/M2 feature set. The full-data recipe uses 100,000 real no-diabetes profiles, every eligible real diabetes profile, and enough synthetic diabetes profiles to make 100,000 diabetes-labeled rows (200,000 rows total). CV folds use the largest available balanced version of that recipe, capped at 100,000 rows per class, because a fold cannot use respondents held out for validation. The synthetic generator sees only the shared model features.

In [4]:
missing_features = sorted(set(FEATURES + [TARGET]) - set(real_pool.columns))
if missing_features:
    raise ValueError(f"Required model columns are missing: {missing_features}")

recipe_preview = pd.DataFrame([
    {"Component": "Real no-diabetes", "Rows in final recipe": REAL_NEGATIVE_CAP,
     "Used for model": "Yes"},
    {"Component": "Real diabetes", "Rows in final recipe": int((real_pool[TARGET] == 1).sum()),
     "Used for model": "Yes"},
    {"Component": "Synthetic diabetes", "Rows in final recipe": BALANCED_TARGET_ROWS // 2 - int((real_pool[TARGET] == 1).sum()),
     "Used for model": "Yes"},
])
display(recipe_preview.style.hide(axis="index"))
print("Model inputs:", FEATURES)

Component,Rows in final recipe,Used for model
Real no-diabetes,100000,Yes
Real diabetes,28078,Yes
Synthetic diabetes,71922,Yes


Model inputs: ['HighBP', 'HighChol', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'Fruits', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Income']


## 7. Define the CART-style synthetic generator

Each generated answer is sampled from a decision tree conditioned on the answers generated earlier in the saved M1/M2 feature order. Leaves retain empirical class probabilities, so the generator samples observed categories instead of rounding arbitrary values into potentially invalid categories. Rows identical to real minority profiles, blocked held-out profiles, or earlier synthetic profiles are rejected. The generator is fitted only on the real diabetes rows supplied to it.

In [5]:
def _row_tuples(frame, columns):
    """Create stable tuples for detecting exact feature-profile copies."""
    return set(map(tuple, frame[columns].astype("int64").to_numpy()))

def generate_cart_synthetic(real_positive, n_new, features, seed=RANDOM_STATE, avoid=None):
    """Generate unique, category-valid minority profiles from training rows only."""
    if n_new <= 0:
        return pd.DataFrame(columns=features)
    real = real_positive[features].astype("int64").reset_index(drop=True)
    if len(real) < 50:
        raise ValueError("CART generator needs at least 50 real diabetes training rows.")

    rng = np.random.default_rng(seed)
    trees = []
    for position, column in enumerate(features):
        parents = features[:position]
        if not parents:
            trees.append((column, parents, None))
            continue
        min_leaf = max(40, min(200, int(len(real) * 0.005)))
        tree = DecisionTreeClassifier(
            max_leaf_nodes=32,
            min_samples_leaf=min_leaf,
            random_state=seed + position,
        )
        tree.fit(real[parents], real[column])
        trees.append((column, parents, tree))

    banned = _row_tuples(real, features)
    if avoid is not None and len(avoid):
        banned |= _row_tuples(avoid, features)
    kept = []
    kept_hashes = set()
    attempts = 0
    have = 0
    while have < n_new and attempts < 30:
        attempts += 1
        batch_size = max(500, int((n_new - have) * 1.35))
        generated = {}
        for column, parents, tree in trees:
            if tree is None:
                levels, counts = np.unique(real[column].to_numpy(), return_counts=True)
                generated[column] = rng.choice(levels, size=batch_size, p=counts / counts.sum())
            else:
                parent_frame = pd.DataFrame({name: generated[name] for name in parents})
                probabilities = tree.predict_proba(parent_frame)
                class_index = (rng.random(batch_size)[:, None] > np.cumsum(probabilities, axis=1)).sum(axis=1)
                class_index = np.minimum(class_index, len(tree.classes_) - 1)
                generated[column] = tree.classes_[class_index]

        candidate = pd.DataFrame(generated, columns=features).astype("int64")
        tuples = list(map(tuple, candidate.to_numpy()))
        keep_mask = []
        for row in tuples:
            accepted = row not in banned and row not in kept_hashes
            keep_mask.append(accepted)
            if accepted:
                kept_hashes.add(row)
        candidate = candidate.loc[keep_mask]
        if not candidate.empty:
            needed = n_new - have
            candidate = candidate.iloc[:needed]
            kept.append(candidate)
            have += len(candidate)

    if have < n_new:
        raise RuntimeError(f"Generator produced only {have:,} of {n_new:,} required unique profiles.")
    return pd.concat(kept, ignore_index=True).iloc[:n_new].reset_index(drop=True)

def make_cart_200k_training(real_training, features, seed=RANDOM_STATE, avoid=None):
    """Apply the 100k real-negative + real/synthetic-positive training recipe."""
    negative = real_training.loc[real_training[TARGET] == 0]
    positive = real_training.loc[real_training[TARGET] == 1]
    n_negative = min(REAL_NEGATIVE_CAP, len(negative))
    if n_negative == 0 or positive.empty:
        raise ValueError("Both classes are required to construct the candidate training set.")
    negative_sample = negative.sample(n=n_negative, random_state=seed)
    # Keep each CV fold balanced at its available negative-row count; the full pool reaches 100k per class.
    positive_target = n_negative
    synthetic_n = positive_target - len(positive)
    if synthetic_n < 0:
        raise ValueError("The fold contains more real diabetes rows than the negative-row target.")
    # Block exact profiles from every real training row, plus held-out rows supplied by CV/test.
    blocked_parts = [real_training[features]]
    if avoid is not None and len(avoid):
        blocked_parts.append(avoid[features])
    blocked_profiles = pd.concat(blocked_parts, ignore_index=True).drop_duplicates()
    synthetic = generate_cart_synthetic(
        positive, synthetic_n, features, seed=seed, avoid=blocked_profiles,
    )
    generated_rows = synthetic.copy()
    generated_rows[TARGET] = 1
    training = pd.concat([
        negative_sample[[TARGET] + features],
        positive[[TARGET] + features],
        generated_rows[[TARGET] + features],
    ], ignore_index=True)
    if len(training) != 2 * n_negative:
        raise AssertionError(f"Expected {2 * n_negative:,} balanced rows, got {len(training):,}.")
    return training, synthetic

## 8. Check generated-profile fidelity using training data only

Before the model experiment, reserve part of the eligible real diabetes rows as a fidelity holdout. The CART generator sees only the other real diabetes rows. A separate classifier then tries to tell held-out real profiles from generated profiles. A ROC AUC near 0.50 means this particular discriminator has difficulty separating them; it is not proof of full realism or privacy. Feature summaries help show where the generated answers differ.

In [6]:
real_diabetes = real_pool.loc[real_pool[TARGET] == 1].reset_index(drop=True)
fidelity_fit, fidelity_holdout = train_test_split(
    real_diabetes, test_size=0.25, random_state=RANDOM_STATE, stratify=real_diabetes[TARGET]
)
fidelity_synthetic = generate_cart_synthetic(
    fidelity_fit, len(fidelity_holdout), FEATURES, seed=RANDOM_STATE + 500,
    avoid=pd.concat([real_pool[FEATURES], fidelity_holdout[FEATURES]], ignore_index=True),
)

fidelity_x = pd.concat([
    fidelity_holdout[FEATURES].reset_index(drop=True),
    fidelity_synthetic[FEATURES].reset_index(drop=True),
], ignore_index=True)
fidelity_y = np.r_[np.zeros(len(fidelity_holdout), dtype=int),
                   np.ones(len(fidelity_synthetic), dtype=int)]
fx_train, fx_test, fy_train, fy_test = train_test_split(
    fidelity_x, fidelity_y, test_size=0.30, random_state=RANDOM_STATE, stratify=fidelity_y
)
discriminator = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=5, max_features="sqrt",
    random_state=RANDOM_STATE, n_jobs=-1,
)
discriminator.fit(fx_train, fy_train)
fidelity_auc = roc_auc_score(fy_test, discriminator.predict_proba(fx_test)[:, 1])

fidelity_summary = pd.DataFrame({
    "Feature": FEATURES,
    "Held-out real mean": fidelity_holdout[FEATURES].mean().to_numpy(),
    "CART synthetic mean": fidelity_synthetic[FEATURES].mean().to_numpy(),
    "Held-out real SD": fidelity_holdout[FEATURES].std().to_numpy(),
    "CART synthetic SD": fidelity_synthetic[FEATURES].std().to_numpy(),
})
fidelity_summary["Mean difference (synthetic - real)"] = (
    fidelity_summary["CART synthetic mean"] - fidelity_summary["Held-out real mean"]
)
print(f"Real-versus-synthetic discriminator ROC AUC: {fidelity_auc:.3f} (0.50 is chance level).")
display(fidelity_summary.style.format(precision=3).hide(axis="index"))
print("Interpret this as a limited training-data fidelity check, not proof that synthetic profiles are real people or bias-free.")

Real-versus-synthetic discriminator ROC AUC: 0.623 (0.50 is chance level).


Feature,Held-out real mean,CART synthetic mean,Held-out real SD,CART synthetic SD,Mean difference (synthetic - real)
HighBP,0.756,0.761,0.430,0.426,0.005
HighChol,0.676,0.672,0.468,0.470,-0.005
BMI,31.990,32.493,7.373,7.669,0.504
Stroke,0.096,0.114,0.294,0.318,0.019
HeartDiseaseorAttack,0.226,0.253,0.418,0.435,0.027
Fruits,0.585,0.563,0.493,0.496,-0.022
HvyAlcoholConsump,0.023,0.029,0.149,0.169,0.007
GenHlth,3.331,3.424,1.014,1.007,0.092
MentHlth,4.647,5.475,9.073,9.548,0.828
PhysHlth,8.264,9.425,11.434,11.633,1.161


Interpret this as a limited training-data fidelity check, not proof that synthetic profiles are real people or bias-free.


## 9. Nested, stratified cross-validation on the eligible real rows

The outer folds estimate performance. Within each outer training fold, inner out-of-fold predictions choose that fold’s threshold for the 85% recall goal. The real-only reference is trained on a class-balanced sample of real rows. The CART candidate uses up to 100,000 real no-diabetes rows, all available real diabetes rows in that fold, and synthetic diabetes rows to balance the available negative count (up to 100,000 per class). Synthetic generation and scaling are repeated inside every training fold; no outer-fold labels enter training or threshold selection.

In [7]:
def make_real_balanced_training(real_training, features, seed):
    """Real-only, balanced reference: keep every positive and sample the same number of negatives."""
    negative = real_training.loc[real_training[TARGET] == 0]
    positive = real_training.loc[real_training[TARGET] == 1]
    n = min(len(negative), len(positive))
    if n == 0:
        raise ValueError("Both classes are required for the real-only reference.")
    neg = negative.sample(n=n, random_state=seed)
    return pd.concat([neg[[TARGET] + features], positive[[TARGET] + features]], ignore_index=True)

def fit_real_reference(real_training, features, seed):
    frame = make_real_balanced_training(real_training, features, seed)
    model = make_logreg(features)
    model.fit(frame[features], frame[TARGET])
    return model, len(frame)

def fit_cart_candidate(real_training, features, seed, heldout=None):
    frame, synthetic = make_cart_200k_training(real_training, features, seed, avoid=heldout)
    model = make_logreg(features)
    model.fit(frame[features], frame[TARGET])
    return model, len(frame), len(synthetic)

outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
outer_results = []
outer_oof_cart = np.full(len(real_pool), np.nan)
outer_oof_reference = np.full(len(real_pool), np.nan)
outer_y = real_pool[TARGET].to_numpy()

for outer_fold, (outer_train_idx, outer_valid_idx) in enumerate(outer_cv.split(real_pool[FEATURES], outer_y), start=1):
    outer_train = real_pool.iloc[outer_train_idx].reset_index(drop=True)
    outer_valid = real_pool.iloc[outer_valid_idx].reset_index(drop=True)
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + outer_fold)
    inner_ref_oof = np.full(len(outer_train), np.nan)
    inner_cart_oof = np.full(len(outer_train), np.nan)
    y_outer_train = outer_train[TARGET].to_numpy()

    for inner_fold, (inner_train_idx, inner_valid_idx) in enumerate(inner_cv.split(outer_train[FEATURES], y_outer_train), start=1):
        inner_train = outer_train.iloc[inner_train_idx].reset_index(drop=True)
        inner_valid = outer_train.iloc[inner_valid_idx].reset_index(drop=True)
        fold_seed = RANDOM_STATE + outer_fold * 100 + inner_fold

        reference, _ = fit_real_reference(inner_train, FEATURES, fold_seed)
        inner_ref_oof[inner_valid_idx] = reference.predict_proba(inner_valid[FEATURES])[:, 1]

        cart, _, _ = fit_cart_candidate(
            inner_train, FEATURES, fold_seed,
            heldout=inner_valid[FEATURES],
        )
        inner_cart_oof[inner_valid_idx] = cart.predict_proba(inner_valid[FEATURES])[:, 1]

    if np.isnan(inner_ref_oof).any() or np.isnan(inner_cart_oof).any():
        raise AssertionError("Some inner out-of-fold predictions were not filled.")
    reference_threshold = threshold_for(y_outer_train, inner_ref_oof)
    cart_threshold = threshold_for(y_outer_train, inner_cart_oof)

    reference, reference_rows = fit_real_reference(outer_train, FEATURES, RANDOM_STATE + outer_fold)
    cart, cart_rows, cart_synthetic_rows = fit_cart_candidate(
        outer_train, FEATURES, RANDOM_STATE + outer_fold,
        heldout=outer_valid[FEATURES],
    )
    ref_prob = reference.predict_proba(outer_valid[FEATURES])[:, 1]
    cart_prob = cart.predict_proba(outer_valid[FEATURES])[:, 1]
    outer_oof_reference[outer_valid_idx] = ref_prob
    outer_oof_cart[outer_valid_idx] = cart_prob

    for name, probability, threshold, rows, synthetic_rows in [
        ("Balanced real-only reference", ref_prob, reference_threshold, reference_rows, 0),
        ("CART synthetic 200k recipe", cart_prob, cart_threshold, cart_rows, cart_synthetic_rows),
    ]:
        row = {"Outer fold": outer_fold, "Model": name,
               "Training rows": rows, "Synthetic diabetes rows": synthetic_rows}
        row.update(evaluate(outer_valid[TARGET].to_numpy(), probability, threshold))
        outer_results.append(row)

    print(f"Completed outer fold {outer_fold}/3.")

cv_results = pd.DataFrame(outer_results)
cv_summary = (cv_results.groupby("Model", as_index=False)
              .agg(**{col: (col, "mean") for col in ["Recall", "Specificity", "Precision (balanced test)", "ROC AUC", "Threshold"]},
                   **{"Recall SD": ("Recall", "std"), "Specificity SD": ("Specificity", "std")}))
display(cv_results.style.format(precision=3).hide(axis="index"))
display(cv_summary.style.format(precision=3).hide(axis="index"))
print("All thresholds above came from inner-fold training predictions; the fixed project test labels have not been scored.")

Completed outer fold 1/3.


Completed outer fold 2/3.


Completed outer fold 3/3.


Outer fold,Model,Training rows,Synthetic diabetes rows,Threshold,Accuracy,Recall,Specificity,Precision (balanced test),F1,ROC AUC,Brier score,False negatives,False positives,Share flagged (balanced test)
1,Balanced real-only reference,37438,0,0.395,0.638,0.851,0.606,0.244,0.380,0.808,0.184,1394,24622,0.454
1,CART synthetic 200k recipe,200000,81281,0.358,0.642,0.847,0.611,0.246,0.381,0.806,0.175,1435,24281,0.448
2,Balanced real-only reference,37436,0,0.397,0.636,0.849,0.604,0.243,0.378,0.805,0.184,1413,24745,0.455
2,CART synthetic 200k recipe,200000,81282,0.359,0.638,0.846,0.607,0.244,0.379,0.804,0.176,1443,24534,0.452
3,Balanced real-only reference,37438,0,0.397,0.637,0.851,0.605,0.244,0.379,0.806,0.185,1398,24642,0.454
3,CART synthetic 200k recipe,200000,81281,0.357,0.639,0.846,0.608,0.244,0.379,0.803,0.176,1445,24478,0.451


Model,Recall,Specificity,Precision (balanced test),ROC AUC,Threshold,Recall SD,Specificity SD
Balanced real-only reference,0.850,0.605,0.244,0.806,0.396,0.001,0.001
CART synthetic 200k recipe,0.846,0.609,0.245,0.804,0.358,0.001,0.002


All thresholds above came from inner-fold training predictions; the fixed project test labels have not been scored.


## 10. Finalize the candidate threshold from training-only out-of-fold scores

For the final candidate, select one threshold from the concatenated outer-fold predictions on the eligible real source. These are out-of-fold predictions: every scored person was excluded from the model and synthetic-generator fit that produced that person’s score. The threshold is fixed before the final test comparison.

In [8]:
if np.isnan(outer_oof_cart).any():
    raise AssertionError("Final candidate OOF predictions are incomplete.")
CART_FINAL_THRESHOLD = threshold_for(outer_y, outer_oof_cart)

# Record the training-only CV summary before opening the final test labels.
cv_summary_path = RESULTS / "02_quality_synthetic_200k_cv_summary.csv"
cv_results_path = RESULTS / "02_quality_synthetic_200k_cv_folds.csv"
cv_summary.to_csv(cv_summary_path, index=False)
cv_results.to_csv(cv_results_path, index=False)

print(f"Training-only CART threshold for {TARGET_RECALL:.0%} target recall: {CART_FINAL_THRESHOLD:.4f}")
print(f"Saved CV summaries: {cv_summary_path} and {cv_results_path}")

Training-only CART threshold for 85% target recall: 0.3529
Saved CV summaries: results\02_quality_synthetic_200k_cv_summary.csv and results\02_quality_synthetic_200k_cv_folds.csv


## 11. Build the final 200k candidate using all eligible real training rows

After the CV recipe and threshold are fixed, train one candidate from the full eligible real pool. The fixed test respondents were removed earlier. The generated training frame is checked for exactly 200,000 rows and an equal target count before scoring.

In [9]:
final_cart_frame, final_cart_synthetic = make_cart_200k_training(
    real_pool, FEATURES, seed=RANDOM_STATE + 10_000,
    avoid=fixed_test[FEATURES],
)
final_cart_model = make_logreg(FEATURES)
final_cart_model.fit(final_cart_frame[FEATURES], final_cart_frame[TARGET])

final_train_counts = final_cart_frame[TARGET].value_counts().sort_index()
print("Final candidate training class counts:", final_train_counts.to_dict())
print("Final synthetic diabetes rows:", len(final_cart_synthetic))
assert len(final_cart_frame) == BALANCED_TARGET_ROWS
assert final_train_counts.to_dict() == {0: 100_000, 1: 100_000}
print("Final candidate is trained. No file was written to application/models/.")

Final candidate training class counts: {0: 100000, 1: 100000}
Final synthetic diabetes rows: 71922
Final candidate is trained. No file was written to application/models/.


## 12. One final comparison on the fixed real test set

This is the first cell that uses the fixed test labels for model scoring. M1 and M2 are loaded exactly as saved, including their own thresholds. The new CART candidate uses the threshold selected in section 10. These are the final comparative results; do not use them to alter the candidate and then report them as a fresh test.

In [10]:
X_test = fixed_test[FEATURES]
y_test = fixed_test[TARGET].to_numpy()

m1_probability = m1_bundle["model"].predict_proba(X_test)[:, 1]
m2_probability = m2_bundle["model"].predict_proba(X_test)[:, 1]
cart_probability = final_cart_model.predict_proba(X_test)[:, 1]

final_rows = []
for name, probabilities, threshold in [
    ("M1 saved 70k real", m1_probability, float(m1_bundle["threshold"])),
    ("M2 saved 200k previous synthetic", m2_probability, float(m2_bundle["threshold"])),
    ("M3 new 200k CART synthetic", cart_probability, CART_FINAL_THRESHOLD),
]:
    metrics = evaluate(y_test, probabilities, threshold)
    adjusted_precision, adjusted_share_flagged = prevalence_adjustment(metrics["Recall"], metrics["Specificity"])
    metrics["Precision at 14% prevalence (approx.)"] = adjusted_precision
    metrics["Share flagged at 14% prevalence (approx.)"] = adjusted_share_flagged
    metrics["Model"] = name
    final_rows.append(metrics)

final_results = pd.DataFrame(final_rows)
ordered = ["Model", "Threshold", "Accuracy", "Recall", "Specificity",
           "Precision (balanced test)", "Precision at 14% prevalence (approx.)",
           "Share flagged (balanced test)", "Share flagged at 14% prevalence (approx.)",
           "ROC AUC", "Brier score", "False negatives", "False positives"]
display(final_results[ordered].style.format(precision=3).hide(axis="index"))

final_csv = RESULTS / "02_quality_synthetic_200k_final_test_comparison.csv"
final_results[ordered].to_csv(final_csv, index=False)
print(f"Saved final comparison: {final_csv}")

Model,Threshold,Accuracy,Recall,Specificity,Precision (balanced test),Precision at 14% prevalence (approx.),Share flagged (balanced test),Share flagged at 14% prevalence (approx.),ROC AUC,Brier score,False negatives,False positives
M1 saved 70k real,0.396,0.726,0.850,0.602,0.681,0.258,0.624,0.461,0.800,0.182,1052,2793
M2 saved 200k previous synthetic,0.376,0.724,0.849,0.599,0.679,0.256,0.625,0.463,0.800,0.182,1062,2812
M3 new 200k CART synthetic,0.353,0.722,0.853,0.591,0.676,0.253,0.631,0.471,0.798,0.184,1031,2873


Saved final comparison: results\02_quality_synthetic_200k_final_test_comparison.csv


## 13. Paired uncertainty for the new candidate versus M1

Because both models score the same respondents, use paired bootstrap resamples of test rows. This gives uncertainty intervals for the difference in recall and specificity at each model’s already-fixed threshold. It is descriptive; the test set remains a one-time final comparison.

In [11]:
rng = np.random.default_rng(RANDOM_STATE)
BOOTSTRAPS = 1_000
m1_pred = (m1_probability >= float(m1_bundle["threshold"])).astype(int)
cart_pred = (cart_probability >= CART_FINAL_THRESHOLD).astype(int)
boot_diffs = []
for _ in range(BOOTSTRAPS):
    idx = rng.integers(0, len(y_test), size=len(y_test))
    y_b = y_test[idx]
    m1_b, cart_b = m1_pred[idx], cart_pred[idx]
    positive = y_b == 1
    negative = y_b == 0
    if not positive.any() or not negative.any():
        continue
    recall_m1 = (m1_b[positive] == 1).mean()
    recall_cart = (cart_b[positive] == 1).mean()
    spec_m1 = (m1_b[negative] == 0).mean()
    spec_cart = (cart_b[negative] == 0).mean()
    boot_diffs.append({"Recall difference (CART - M1)": recall_cart - recall_m1,
                       "Specificity difference (CART - M1)": spec_cart - spec_m1})

boot_frame = pd.DataFrame(boot_diffs)
uncertainty_rows = []
for metric in boot_frame.columns:
    values = boot_frame[metric]
    uncertainty_rows.append({"Difference": metric,
                             "Observed difference": (final_results.loc[2, "Recall"] - final_results.loc[0, "Recall"])
                                 if metric.startswith("Recall") else
                                 (final_results.loc[2, "Specificity"] - final_results.loc[0, "Specificity"]),
                             "Paired bootstrap 95% lower": values.quantile(0.025),
                             "Paired bootstrap 95% upper": values.quantile(0.975)})
uncertainty = pd.DataFrame(uncertainty_rows)
display(uncertainty.style.format(precision=3).hide(axis="index"))

Difference,Observed difference,Paired bootstrap 95% lower,Paired bootstrap 95% upper
Recall difference (CART - M1),0.003,-0.001,0.007
Specificity difference (CART - M1),-0.011,-0.016,-0.007


## 14. Decision and limitations

**Observed run (seed 42):**

- The current M1 and M2 bundles both use the same **14 features**: `HighBP`, `HighChol`, `BMI`, `Stroke`, `HeartDiseaseorAttack`, `Fruits`, `HvyAlcoholConsump`, `GenHlth`, `MentHlth`, `PhysHlth`, `DiffWalk`, `Sex`, `Age`, and `Income`.
- After removing all **14,039 exact test rows** from the 229,474-row unbalanced source, the eligible training pool had **187,357 no-diabetes** and **28,078 diabetes** respondents.
- The new full recipe used 100,000 real no-diabetes rows, 28,078 real diabetes rows, and 71,922 CART-generated diabetes profiles. The generator’s held-out real-versus-synthetic discriminator ROC AUC was **0.623**, so the synthetic profiles were still distinguishable from held-out real profiles.
- In nested training-only CV, the CART recipe averaged **84.6% recall** and **60.9% specificity**; the balanced real-only reference averaged **85.0% recall** and **60.5% specificity**. The CV advantage was small and did not establish a reliable improvement.
- On the final fixed test, M1 had **85.01% recall**, **60.21% specificity**, and **68.12% balanced-test precision**. The new CART candidate had **85.31% recall**, **59.07% specificity**, and **67.58% balanced-test precision**. Its approximate precision at 14% prevalence was **25.34%**, versus M1’s **25.81%**.
- CART minus M1 specificity was **−1.14 percentage points** (paired bootstrap 95% interval: **−1.63 to −0.67 points**). Recall changed by **+0.30 points** (95% interval: **−0.13 to +0.69 points**).

**Decision:** the candidate barely clears the recall requirement but reduces specificity and precision, with the specificity difference consistently unfavorable in the paired bootstrap. **Keep M1 as the app model.** This experiment does not support adding more synthetic diabetes rows as a performance improvement.

**Limitations:** the data are self-reported 2015 U.S. survey responses; the target is reported diagnosed diabetes; generated profiles are not real respondents; an adversarial classifier only checks detectable differences in this feature set; test precision is measured on a balanced sample; the 14% adjustment is approximate; and performance is not clinical validation. A synthetic-data recipe can preserve broad patterns while still creating biased or implausible combinations.

**Next step:** retain M1. Any later candidate should be selected through training-only validation and judged against a new external dataset if one becomes available; do not tune this candidate using the final test results.